# 02b — Retrain the 2 Transformers on Clean Split + Metrics

Runs **Swin-Base and DeiT-Base** on the same clean split (AdamW 1e-5, cosine, label smoothing 0.1). These are the slow models — if the session nears the ~9hr limit, Swin is already saved before DeiT starts, so you can re-run from DeiT alone.

Each saves `*_clean.pth` + `*_preds.npz` to `/kaggle/working/`. Combined table is in **02c**.

**Setup:** attach (1) the dataset and (2) notebook 01's output; set `SPLIT_DIR` below.

In [1]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
# Path where notebook 01's .npy outputs are mounted (adjust to your dataset name):
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f'Missing path: {pth}'
print('Paths OK')

Paths OK


In [2]:
!pip install timm --quiet
import numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, roc_auc_score)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# --- load the clean split (single source of truth) ---
train_idx = np.load(f'{SPLIT_DIR}/clean_train_indices.npy').tolist()
val_idx   = np.load(f'{SPLIT_DIR}/clean_val_indices.npy').tolist()
test_idx  = np.load(f'{SPLIT_DIR}/clean_test_indices.npy').tolist()
class_names = open(f'{SPLIT_DIR}/class_names.txt').read().splitlines()
print(f'Train {len(train_idx)}  Val {len(val_idx)}  Test {len(test_idx)}')
print('Classes:', class_names)
SEVERE_IDX = class_names.index('Severe')
print('Severe class index:', SEVERE_IDX)

Device: cuda
Train 2485  Val 533  Test 536
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Severe class index: 4


In [3]:
# ============================================================
# Shared transforms + loader factory
# Augmentation ON for training (defensible on the smaller clean set).
# `size` varies per model (299 Inception, 224 the rest).
# ============================================================
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)),
        transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM,
    ])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, crop_from=None):
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  val_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True,  num_workers=0),
            DataLoader(val_ds,   BATCH, shuffle=False, num_workers=0),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=0))

In [4]:
# ============================================================
# Shared train / evaluate / save helpers
# ============================================================
RESULTS = {}  # model_name -> metrics dict

def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        running = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):        # Inception aux logits
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,'logits') else out, y)
            loss.backward(); optimizer.step()
            running += loss.item()
        if scheduler: scheduler.step()
        print(f'  epoch {ep+1}/{epochs}  loss {running/len(loader):.4f}')
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(device)
        out = model(x)
        out = out.logits if hasattr(out,'logits') else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def report_and_save(name, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weap  = f1_score(labels, preds, average='weighted')
    pr, rc, f1, sup = precision_recall_fscore_support(labels, preds, labels=range(NUM_CLASSES), zero_division=0)
    try:
        auc_macro = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
        auc_per = roc_auc_score(np.eye(NUM_CLASSES)[labels], probs, multi_class='ovr', average=None)
    except Exception as e:
        print('  AUC warning:', e); auc_macro, auc_per = float('nan'), [float('nan')]*NUM_CLASSES
    print(f'\n=== {name} ===')
    print(f'Accuracy {acc:.4f} | Macro-F1 {f1_macro:.4f} | Weighted-F1 {f1_weap:.4f} | Macro-AUC {auc_macro:.4f}')
    print(f"{'Class':<16}{'Prec':>7}{'Rec':>7}{'F1':>7}{'AUC':>7}{'N':>6}")
    for c, cn in enumerate(class_names):
        star = '  <-- SEVERE' if c == SEVERE_IDX else ''
        print(f'{cn:<16}{pr[c]:>7.3f}{rc[c]:>7.3f}{f1[c]:>7.3f}{auc_per[c]:>7.3f}{sup[c]:>6}{star}')
    # save weights already done by caller; save preds here
    np.savez(f'{OUT_DIR}/{name}_preds.npz', probs=probs, preds=preds, labels=labels)
    RESULTS[name] = dict(acc=acc, f1_macro=f1_macro, f1_weighted=f1_weap,
                         auc_macro=auc_macro, severe_recall=rc[SEVERE_IDX],
                         severe_f1=f1[SEVERE_IDX])
    print(f'  saved {OUT_DIR}/{name}_preds.npz')

## Model 1 — Swin-Base (slow). timm, AdamW 1e-5, cosine, label smoothing.

In [5]:
tr, va, te = make_loaders(224)
swin = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=NUM_CLASSES).to(device)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
opt  = optim.AdamW(swin.parameters(), lr=1e-5, weight_decay=0.05)
sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
print('Training Swin-Base...')
swin = train_model(swin, tr, crit, opt, sch)
torch.save(swin.state_dict(), f'{OUT_DIR}/swin_base_clean.pth')
probs, labels = evaluate(swin, te)
report_and_save('swin_base', probs, labels)
del swin; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Training Swin-Base...
  epoch 1/10  loss 0.7002
  epoch 2/10  loss 0.4655
  epoch 3/10  loss 0.4436
  epoch 4/10  loss 0.4375
  epoch 5/10  loss 0.4272
  epoch 6/10  loss 0.4212
  epoch 7/10  loss 0.4195
  epoch 8/10  loss 0.4150
  epoch 9/10  loss 0.4114
  epoch 10/10  loss 0.4161

=== swin_base ===
Accuracy 0.9664 | Macro-F1 0.9621 | Weighted-F1 0.9658 | Macro-AUC 0.9979
Class              Prec    Rec     F1    AUC     N
Mild              0.963  0.987  0.975  0.999    79
Moderate          0.937  0.983  0.959  0.997    60
No_DR             0.980  1.000  0.990  1.000   146
Proliferate_DR    0.953  0.982  0.967  0.996   164
Severe            1.000  0.851  0.919  0.997    87  <-- SEVERE
  saved /kaggle/working/swin_base_preds.npz


In [6]:
import os
print([f for f in os.listdir('/kaggle/working') if f.endswith(('.pth','.npz'))])

['swin_base_preds.npz', 'swin_base_clean.pth']


## Model 2 — DeiT-Base (slow). timm, AdamW 1e-5, cosine, label smoothing.

In [7]:
tr, va, te = make_loaders(224)
deit = timm.create_model('deit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES).to(device)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
opt  = optim.AdamW(deit.parameters(), lr=1e-5, weight_decay=0.05)
sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
print('Training DeiT-Base...')
deit = train_model(deit, tr, crit, opt, sch)
torch.save(deit.state_dict(), f'{OUT_DIR}/deit_base_clean.pth')
probs, labels = evaluate(deit, te)
report_and_save('deit_base', probs, labels)
del deit; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Training DeiT-Base...
  epoch 1/10  loss 0.6575
  epoch 2/10  loss 0.4525
  epoch 3/10  loss 0.4380
  epoch 4/10  loss 0.4261
  epoch 5/10  loss 0.4181
  epoch 6/10  loss 0.4130
  epoch 7/10  loss 0.4121
  epoch 8/10  loss 0.4055
  epoch 9/10  loss 0.4076
  epoch 10/10  loss 0.4056

=== deit_base ===
Accuracy 0.9646 | Macro-F1 0.9587 | Weighted-F1 0.9643 | Macro-AUC 0.9985
Class              Prec    Rec     F1    AUC     N
Mild              0.929  0.987  0.957  0.999    79
Moderate          0.908  0.983  0.944  0.996    60
No_DR             0.993  1.000  0.997  1.000   146
Proliferate_DR    0.963  0.963  0.963  0.997   164
Severe            1.000  0.874  0.933  1.000    87  <-- SEVERE
  saved /kaggle/working/deit_base_preds.npz


### Per-session mini-summary (these 2 models)

In [8]:
import pandas as pd
df = pd.DataFrame(RESULTS).T
print('Models trained this session:', list(RESULTS.keys()))
print(df.round(4).to_string())
print('\nWeights + .npz saved. Save Version (Commit), then run 02c to combine all 5.')

Models trained this session: ['swin_base', 'deit_base']
              acc  f1_macro  f1_weighted  auc_macro  severe_recall  severe_f1
swin_base  0.9664    0.9621       0.9658     0.9979         0.8506     0.9193
deit_base  0.9646    0.9587       0.9643     0.9985         0.8736     0.9325

Weights + .npz saved. Save Version (Commit), then run 02c to combine all 5.
